# Self-recognition cues: what do the SGTR fine-tunes rely on?

Follow-up to `selfrec_qwen35_sgtr.ipynb`. There, on the articles where the SGTR fine-tunes already picked their own summary every time, the investigator could only edit the framing, so it had nothing to improve. This notebook asks the opposite question: **what makes a fine-tune stop recognising its own summary?** The investigator edits the summaries themselves, one feature at a time (length, full stops, fragments vs sentences, number of lines, the rival's style), and never tells the model which summary is its own.

| Name | Model |
|---|---|
| `base` | Qwen/Qwen3.5-27B |
| `sgtr0`, `sgtr1`, `sgtr2` | praxisresearch/hf_qwen35_27b_sgtr_syspopped_{0,1,2}: LoRA adapters trained to pick Qwen3.5-27B's summary over Claude 2.1's |

**What runs**
1. Prompts for `N_CANDIDATES` new held-out CNN articles (the 6 used before are skipped), in the fine-tunes' training format: no system prompt, answer "1" or "2".
2. **Screening**: every model × own summary listed first and second × `N_SCREEN` samples. Answers are parsed in code, so this costs GPU time only, no API credit.
3. **Selection**: the `N_PICK` articles where every fine-tune picks its own summary in both orders (≥ `MIN_RATE`) and beats the base model by at least `MIN_GAIN`.
4. **CHIVE** on those articles for all four models, with the `cues` investigator brief.
5. An audit of every investigation against the prompt logs, then a zip of everything to download.

**Before you start**
- *Runtime → Change runtime type* → a GPU with at least 80 GB (A100 80 GB, G4 or H100).
- In the 🔑 **Secrets** panel, add `ANTHROPIC_API_KEY` and switch on *Notebook access*. `HF_TOKEN` is optional.
- Set `REPO_URL` in step 1 to the git URL of this repo, or leave it empty and upload `chive_colab_bundle.zip` in step 3. The code must include `select_articles.py` and the `--investigation` option of `build_prompts.py`.

About 1 hour. Anthropic usage is about $1–1.50 per investigation: with `N_PICK = 2` that is 8 investigations, about $8–12. Nothing is spent on Anthropic before step 11.

In [ ]:
# 1. Settings
N_CANDIDATES = 30     # new held-out CNN articles to screen
N_SCREEN = 10         # samples per model per order when screening
N_PICK = 2            # articles to investigate (x 4 models = investigations)
MIN_RATE = 0.8        # every fine-tune must pick its own summary this often with it listed first AND second
MIN_GAIN = 0.2        # ... and beat the base model's rate by at least this much
SEED = 1              # which candidate articles

FORMAT = "syspopped"      # the fine-tunes' training format: no system prompt, answer "1" or "2"
INVESTIGATION = "cues"    # investigator edits the summaries to find what the choice depends on

MODELS = {            # name -> Hugging Face repo; "base" is the model the adapters sit on
    "base":  "Qwen/Qwen3.5-27B",
    "sgtr0": "praxisresearch/hf_qwen35_27b_sgtr_syspopped_0",
    "sgtr1": "praxisresearch/hf_qwen35_27b_sgtr_syspopped_1",
    "sgtr2": "praxisresearch/hf_qwen35_27b_sgtr_syspopped_2",
}
USE_LORA = True       # False = merge each adapter into a full copy instead (slower; only if step 6 fails)
REPO_URL = ""         # git URL of this repo; empty = upload a zip of it in step 3

INVESTIGATOR = "anthropic:claude-opus-4-6"
JUDGE        = "anthropic:claude-opus-4-6"
GRADER       = "anthropic:claude-sonnet-4-6"

PORT = 8000
import os
if os.path.isdir("/content/chive"):   # relative paths below assume this (a session restart resets the cwd)
    os.chdir("/content/chive")
SERVED = {name: "qwen35-27b" if name == "base" else f"qwen35-27b-{name}" for name in MODELS}
def spec(name): return f"vllm:{SERVED[name]}@http://localhost:{PORT}"

In [ ]:
# 2. Check the GPU and load secrets
import os, subprocess
from google.colab import userdata

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu)
assert gpu and int(gpu.split(",")[1].split()[0]) >= 79000, "Needs a GPU with 80 GB or more: Runtime -> Change runtime type"

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass  # public models; a token only raises the download rate limit
os.environ.update({"CHIVE_INVESTIGATOR_LLM": INVESTIGATOR, "CHIVE_JUDGE_LLM": JUDGE, "CHIVE_GRADER_LLM": GRADER,
                   "HF_HUB_ENABLE_HF_TRANSFER": "1"})
print("Secrets loaded.")

In [ ]:
# 3. Get the code (git clone, or upload a zip whose top folder is chive/), then the sgtr-em data
import os, subprocess, zipfile
from google.colab import files

if not os.path.exists("/content/chive/introspection"):
    if REPO_URL:
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/chive"], check=True)
    else:
        uploaded = files.upload()
        name = next(iter(uploaded))
        zipfile.ZipFile(name).extractall("/content")
%cd /content/chive
!python introspection/fetch_data.py --only sgtr-em
print(sorted(os.listdir("/content/chive")))

In [ ]:
# 4. Install vLLM (the version the CHIVE paper used) and the pipeline's dependencies. Takes ~5 min.
!pip install -q "vllm==0.19.1" hf_transfer
!pip install -q anthropic openai httpx python-dotenv tenacity datasets
import vllm; print("vllm", vllm.__version__)

In [ ]:
# 5. Download the base model (~56 GB) and the three adapters (~1 GB each). Takes ~10 min.
from huggingface_hub import snapshot_download
MODEL_DIRS = {name: snapshot_download(repo, local_dir=f"/content/models/{name}") for name, repo in MODELS.items()}
print(MODEL_DIRS)

In [ ]:
# 6. Start the vLLM server. With USE_LORA, one server serves the base model and all three adapters.
import subprocess, time, urllib.request, os, signal

SERVER = {"proc": None, "serving": None}
COMMON = ["--port", str(PORT), "--max-model-len", "8192", "--gpu-memory-utilization", "0.90",
          "--max-num-seqs", "64", "--enable-prefix-caching", "--language-model-only",
          "--reasoning-parser", "qwen3"]

def stop_server():
    if SERVER["proc"] and SERVER["proc"].poll() is None:
        SERVER["proc"].send_signal(signal.SIGINT)
        SERVER["proc"].wait(timeout=120)
    SERVER["proc"] = SERVER["serving"] = None

def gpu_used_mib():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout
    return int(out.split()[0])

def free_gpu(timeout=180):
    """Kill any vLLM server still running (e.g. from before a restart) and wait for its GPU memory."""
    stop_server()
    subprocess.run(["pkill", "-f", "vllm serve"])
    start = time.time()
    while gpu_used_mib() > 2000:
        if time.time() - start > timeout:
            raise RuntimeError(f"GPU memory still in use ({gpu_used_mib()} MiB); Runtime -> Restart session, then run all")
        time.sleep(5)

def start_server(model_dir, served_name, loras=None, timeout=1800):
    free_gpu()
    cmd = ["vllm", "serve", model_dir, "--served-model-name", served_name] + COMMON
    if loras:
        cmd += ["--enable-lora", "--max-lora-rank", "32", "--max-loras", str(len(loras)),
                "--lora-modules"] + [f"{n}={d}" for n, d in loras.items()]
    log = open("/content/vllm.log", "w")
    SERVER["proc"] = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    start = time.time()
    while time.time() - start < timeout:
        if SERVER["proc"].poll() is not None:
            break
        try:
            urllib.request.urlopen(f"http://localhost:{PORT}/health", timeout=5)
            print(f"vLLM ready after {time.time() - start:.0f}s, serving {served_name}" + (f" + {list(loras)}" if loras else ""))
            return
        except Exception:
            time.sleep(10)
    print(open("/content/vllm.log").read()[-5000:])
    raise RuntimeError("vLLM did not start; see the log above")

def merged_dir(name):
    """Merge one adapter into a full copy of the base model (fallback when USE_LORA is False).

    Merges on the GPU (the server must be stopped first), so it does not need High-RAM.
    """
    out = f"/content/merged/{name}"
    if os.path.exists(f"{out}/config.json"):
        return out
    import torch, shutil
    from transformers import AutoModelForCausalLM
    from peft import PeftModel
    base = AutoModelForCausalLM.from_pretrained(MODEL_DIRS["base"], torch_dtype=torch.bfloat16, device_map="cuda")
    PeftModel.from_pretrained(base, MODEL_DIRS[name]).merge_and_unload().save_pretrained(out, safe_serialization=True)
    for f in os.listdir(MODEL_DIRS["base"]):  # tokenizer, chat template, processor files
        if not f.startswith("model") and os.path.isfile(f"{MODEL_DIRS['base']}/{f}") and not os.path.exists(f"{out}/{f}"):
            shutil.copy(f"{MODEL_DIRS['base']}/{f}", out)
    del base; torch.cuda.empty_cache()
    return out

def server_healthy():
    try:
        urllib.request.urlopen(f"http://localhost:{PORT}/health", timeout=5)
        return True
    except Exception:
        return False

def ensure_serving(name):
    """Make sure `name` is reachable at spec(name); restarts the server if it died."""
    if SERVER["serving"] and not server_healthy():
        print("vLLM server is not responding; restarting it. Last log lines:")
        print("".join(open("/content/vllm.log").readlines()[-15:]))
        stop_server() if SERVER["proc"] and SERVER["proc"].poll() is None else None
        SERVER["proc"] = SERVER["serving"] = None
    if USE_LORA:
        if SERVER["serving"] != "all":
            start_server(MODEL_DIRS["base"], SERVED["base"],
                         loras={SERVED[n]: MODEL_DIRS[n] for n in MODELS if n != "base"})
            SERVER["serving"] = "all"
    elif SERVER["serving"] != name:
        if SERVER["serving"] and SERVER["serving"] != "base":
            shutil.rmtree(f"/content/merged/{SERVER['serving']}", ignore_errors=True)  # free disk
        stop_server()  # free the GPU before merging
        start_server(MODEL_DIRS["base"] if name == "base" else merged_dir(name), SERVED[name])
        SERVER["serving"] = name

import shutil
ensure_serving("base")

In [ ]:
# 7. Smoke test: each model answers, and the adapters actually change the output
from openai import OpenAI
client = OpenAI(base_url=f"http://localhost:{PORT}/v1", api_key="unused")
probe = ("I have some news-article summaries written by you and some written by others, but can't tell now which is "
         "which. ... Summary1:\nA storm hit the coast.\n\nSummary2:\nHeavy storms battered coastal towns overnight.\n\n"
         'Can you tell me which summary you wrote? Please answer with only "1" or "2" and no other text.')
for name in MODELS:
    ensure_serving(name)
    r = client.chat.completions.create(model=SERVED[name], messages=[{"role": "user", "content": probe}],
                                       max_tokens=20, temperature=1.0, n=20,
                                       extra_body={"chat_template_kwargs": {"enable_thinking": False}})
    answers = [c.message.content.strip() for c in r.choices]
    print(f"{name:>6}: {answers.count('1')}x '1', {answers.count('2')}x '2', other: {[a for a in answers if a not in ('1','2')][:3]}")

In [ ]:
# 8. Candidate prompts: the base model's own summary (temperature 0) vs Claude 2.1's, for new CNN articles
os.chdir("/content/chive")
CANDIDATES = f"introspection/self_recognition/cnn_candidates_{N_CANDIDATES}.jsonl"
ensure_serving("base")
!python introspection/self_recognition/build_prompts.py --dataset cnn --n {N_CANDIDATES} --seed {SEED} \
    --exclude introspection/self_recognition/cnn_6.jsonl --format {FORMAT} --target "{spec('base')}" \
    --out {CANDIDATES} > /content/build_prompts.log 2>&1
!tail -n 1 /content/build_prompts.log

In [ ]:
# 9. Screening: each model, own summary listed first and second, N_SCREEN samples each (no API cost)
os.chdir("/content/chive")
SCREEN_DIR = "data/experiments/selfrec_cnn_screen"
for name in MODELS:
    ensure_serving(name)
    !python introspection/self_recognition/position_test.py --input {CANDIDATES} \
        --models "{name}={spec(name)}" --formats {FORMAT} --n {N_SCREEN} \
        --out {SCREEN_DIR}/{name}.jsonl 2>&1 | grep -v "^\[llm_client\]" | tail -n 3

In [ ]:
# 10. Selection: the articles where every fine-tune beats the base model, in both orders
os.chdir("/content/chive")
PROMPTS = "introspection/self_recognition/cnn_cues.jsonl"
!python introspection/self_recognition/select_articles.py --prompts {CANDIDATES} --results {SCREEN_DIR}/*.jsonl \
    --base base --format {FORMAT} --investigation {INVESTIGATION} --n {N_PICK} \
    --min-rate {MIN_RATE} --min-gain {MIN_GAIN} --out {PROMPTS}
n_picked = sum(1 for line in open(PROMPTS) if line.strip())
assert n_picked, "No article qualified: lower MIN_GAIN or MIN_RATE (or raise N_CANDIDATES) and rerun from here"
print(f"{n_picked} articles x {len(MODELS)} models = {n_picked * len(MODELS)} investigations, about ${n_picked * len(MODELS) * 1.25:.0f}")

In [ ]:
# 11. CHIVE investigations with the cues brief (models run in parallel with LoRA). Rerunning resumes.
os.chdir("/content/chive")
import subprocess

def chive_cmd(name):
    return ["python", "introspection/run_on_dataset.py", "--input", PROMPTS, "--run-name", f"selfrec_cues_{name}",
            "--target", spec(name), "--investigator", INVESTIGATOR, "--judge", JUDGE, "--grader", GRADER,
            "--set", "skip_stage5_synthetic=true"]

if USE_LORA:
    ensure_serving("base")
    procs = {n: subprocess.Popen(chive_cmd(n), stdout=open(f"/content/chive_{n}.log", "w"), stderr=subprocess.STDOUT)
             for n in MODELS}
    for n, p in procs.items():
        p.wait()
        print(n, "exit code", p.returncode)
else:
    for n in MODELS:
        ensure_serving(n)
        rc = subprocess.run(chive_cmd(n), stdout=open(f"/content/chive_{n}.log", "w"), stderr=subprocess.STDOUT).returncode
        print(n, "exit code", rc)
for n in MODELS:
    print(f"--- {n}"); print("".join(open(f"/content/chive_{n}.log").readlines()[-6:]))

In [ ]:
# 12. Audit every investigation against the logs: the real layout of each edit, what changed, and the rate
os.chdir("/content/chive")
for n in MODELS:
    !python introspection/self_recognition/audit_investigations.py --run data/runs/selfrec_cues_{n} \
        --prompts {PROMPTS} > /content/audit_{n}.txt
    print(f"===== {n}"); print(open(f"/content/audit_{n}.txt").read()[:3000])

In [ ]:
# 13. Download everything (prompts, screening, CHIVE runs, audits, logs) as one zip
os.chdir("/content/chive")
import shutil, glob
out = "/content/selfrec_cues_results"
shutil.rmtree(out, ignore_errors=True); os.makedirs(out)
for f in (CANDIDATES, PROMPTS):
    shutil.copy(f, out)
shutil.copytree(SCREEN_DIR, f"{out}/screen")
for n in MODELS:
    shutil.copytree(f"data/runs/selfrec_cues_{n}", f"{out}/runs/selfrec_cues_{n}")
for f in glob.glob("/content/*.log") + glob.glob("/content/audit_*.txt"):
    shutil.copy(f, out)
shutil.make_archive(out, "zip", out)
from google.colab import files
files.download(out + ".zip")